# PyTorch DAGs, Gradients & Detachment - Quick Reference

## TL;DR
This notebook covers PyTorch's automatic differentiation system:
- **Computation Graph (DAG)**: How PyTorch tracks operations for backprop
- **Gradient Tracking**: `requires_grad`, `torch.no_grad()`, `.backward()`
- **Tensor Detachment**: `.detach()` for breaking gradient flow and memory management
- **Autograd System**: Understanding leaf vs non-leaf tensors

In [ ]:
import torch
import numpy as np

print(f"PyTorch version: {torch.__version__}")
torch.manual_seed(42)  # For reproducibility

## Computation Graph (DAG)

PyTorch builds a Directed Acyclic Graph to track operations for automatic differentiation:
- **Leaf tensors**: Created by user (e.g., `torch.tensor(..., requires_grad=True)`)
- **Non-leaf tensors**: Result of operations, have `grad_fn` showing the operation

In [ ]:
# Build computation graph
x = torch.tensor([[1.0, 2.0]], requires_grad=True)
y = torch.tensor([[3.0, 4.0]], requires_grad=True)

# Check leaf tensor properties
print(f"x.is_leaf: {x.is_leaf}")        # True - created by user
print(f"x.grad_fn: {x.grad_fn}")        # None - leaf tensor

# Operations create graph nodes
z = x + y                                # AddBackward
w = z * 2                                # MulBackward  
result = w.sum()                         # SumBackward

print(f"z.grad_fn: {z.grad_fn}")
print(f"result.grad_fn: {result.grad_fn}")
print(f"result.is_leaf: {result.is_leaf}")  # False - computed tensor

## Gradient Tracking

Control when and how gradients are computed:
- **`requires_grad=True`**: Enable gradient tracking for specific tensors
- **`torch.no_grad()`**: Temporarily disable gradients for ALL operations in context
- **`.backward()`**: Compute gradients through the computation graph

In [ ]:
# Enable/disable gradient tracking
x = torch.randn(2, 2)                    # Default: requires_grad=False
x_grad = torch.randn(2, 2, requires_grad=True)  # Explicit: requires_grad=True
x.requires_grad_(True)                   # In-place enable

# Disable gradients temporarily
with torch.no_grad():
    y = x * 2                            # y.requires_grad = False

# Compute gradients
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x.pow(2).sum()                       # y = x₁² + x₂²
y.backward()                             # Compute ∂y/∂x

print(f"x.grad: {x.grad}")              # [4.0, 6.0] = [2*x₁, 2*x₂]

## Tensor Detachment

**What `.detach()` does:**
- Creates new tensor pointing to same memory location (shares data)
- Breaks connection to computation graph (`grad_fn=None`, `requires_grad=False`)
- Original tensor's graph remains intact

**Key differences:**
- **`.detach()`**: Affects specific tensor, shares memory
- **`torch.no_grad()`**: Affects all operations in context, no memory sharing
- **No "reattachment"**: Once detached, original computation history is lost forever

In [ ]:
# .detach() breaks gradient flow, shares memory
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x * 2

# Detachment methods
detached = y.detach()                    # Shares memory, no gradients
cloned = y.clone()                       # New memory, keeps gradients  
detached_cloned = y.detach().clone()     # New memory, no gradients

print(f"y.requires_grad: {y.requires_grad}")           # True
print(f"detached.requires_grad: {detached.requires_grad}")  # False
print(f"Same memory: {y.data_ptr() == detached.data_ptr()}")  # True

# Memory sharing demonstration - modifying one affects the other
detached[0] = 999.0                      # Change detached tensor
print(f"Original y after modifying detached: {y}")  # y[0] is now 999!

# "Reattachment" - can enable gradients but loses original history
reattached = detached.requires_grad_(True)  # Enable gradients on detached tensor
# But original computation graph connection is lost forever

# Common use cases:
# 1. NumPy conversion: tensor.detach().numpy()
# 2. Stop gradient flow: loss = criterion(pred, target.detach())
# 3. Memory optimization: break graph references
# 4. Safe copy: tensor.detach().clone()  # New memory + no gradients

## Practical Example: Training Step

**Key practices:**
- Use `torch.no_grad()` during parameter updates (no gradients needed)
- Clear gradients with `.zero_()` between iterations (gradients accumulate by default)

In [ ]:
# Simple neural network training step
W = torch.randn(3, 2, requires_grad=True)  # Weights
b = torch.randn(3, requires_grad=True)     # Bias
x = torch.randn(5, 2)                      # Input batch

# Forward pass
y = torch.matmul(x, W.T) + b               # Linear layer
z = torch.relu(y)                          # Activation
target = torch.randn(5, 3)
loss = torch.mean((z - target) ** 2)       # MSE loss

# Backward pass
loss.backward()                            # Compute gradients

# Optimizer step (manual gradient descent)
lr = 0.01
with torch.no_grad():                      # Disable gradients for updates
    W -= lr * W.grad
    b -= lr * b.grad

# Clear gradients for next iteration
W.grad.zero_()
b.grad.zero_()

print(f"Loss: {loss.item():.4f}")
print("Training step completed")